# PRJEB30386 — эталон после аннотации → PCR1 → фрагментация → PCR2 → NovaSeq PE150

Биологический эталон строится только из `post_annotation_filtered/airr_pass`. Симуляция сохраняет наблюдаемую нуклеотидную последовательность V…J, включая реальные SHM, использует численность UMI как число исходных копий, добавляет общую модель ошибок InSilicoSeq NovaSeq.

**Фрагментация:** `uniform single random-cut`.


## Выходные данные

Имя ветки собирается в ячейке параметров из базового имени, режима эталона, `LIBRARY_INPUT_NG` и `READ_BUDGET_MULTIPLIER`, например:

`results/PRJEB30386/simulated/<base>_amp_umicons_in5ng_rb3x_ss250/`

Каталог `qc/` предназначен для контроля качества этой симуляции.


## 1. Окружение

In [ ]:
import os, sys, sysconfig, subprocess, time, gzip, csv, math, re, shutil, json, hashlib
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np

_ENV_CANDIDATES = [
    os.environ.get("BCR_ENV", ""),
    os.environ.get("CONDA_PREFIX", ""),
    "/data/user/epishkin/conda/envs/bcr_env",
    "/opt/conda/envs/bcr_env",
    "/Users/epishkin/mamba/envs/bcr_env",
]
_CONDA_ENV = next((p for p in _ENV_CANDIDATES if p and os.path.isdir(str(Path(p) / "bin"))), None)
if not _CONDA_ENV:
    raise FileNotFoundError("bcr_env not found; activate it or set BCR_ENV")

os.environ["PATH"] = str(Path(_CONDA_ENV) / "bin") + ":" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
for _site in [
    str(Path(_CONDA_ENV) / "lib/python3.11/site-packages"),
    str(Path(_CONDA_ENV) / "lib/python3.12/site-packages"),
    sysconfig.get_path("purelib"),
]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)

# Для ветви NovaSeq требуется только InSilicoSeq; отсутствие сторонних
# инструментов выравнивания не является ошибкой.
for tool in ("iss",):
    p = shutil.which(tool)
    if not p:
        raise RuntimeError(f"Required tool not found: {tool}")
    print(f"{tool}: {p}")
print("numpy:", np.__version__)


## 2. Параметры

In [ ]:
BRANCH_NAME = "insilicoseq_150bp_novaseq_post_annotation_filtered_ultrasonic_like"

# ---------------------------------------------------------------------
# Эталон, входная порция библиотеки и глубина (входят в имя ветки)
# ---------------------------------------------------------------------
# amplicon_with_primers: физический ампликон PCR1 = V-leader праймер + склеенный
#   рид (остаток leader, V…J, C-фрагмент) + revcomp(C-праймер). Праймеры вырезаны
#   на стадии pr_trimmed, их имена восстанавливаются из заголовков pr_trimmed.
# vj_only: эталон обрезан ровно до V…J.
TRUTH_TEMPLATE_MODE = "amplicon_with_primers"
# umi_consensus: каждый UMI = одна исходная молекула; её V…J — поколоночный консенсус
#   ридов UMI модального локуса и модальной длины (>= MIN_UMI_CONSENSUS_READS ридов,
#   без ничьих). UMI без консенсуса (1 рид, ничья, мало ридов одной длины) симулируются
#   самым частым ридом, но помечаются verified=False и не входят в оценку сборщиков.
UMI_COLLAPSE_MODE = "umi_consensus"
MIN_UMI_CONSENSUS_READS = 2
# Масса PCR1-продукта, идущая на фрагментацию (протокол 5′MTPX: 5–10 нг). Число
# молекул порции = масса / (средняя длина ампликона · 650 г/моль) · N_A, то есть
# тысячи копий каждой исходной молекулы, как в реальной подготовке библиотеки.
LIBRARY_INPUT_NG = 5.0
# Множитель глубины относительно суммы сырых пар исходных библиотек: исходный
# MiSeq читал неразрезанные ампликоны, после фрагментации на молекулу нужно
# несколько пар.
READ_BUDGET_MULTIPLIER = 3.0

# Мягкий size-selection после фрагментации: центр распределения длин вставок (нт).
# Входит в имя ветки. 250 нт для ~430–500-нт ампликона даёт перекрывающиеся 5′/3′-куски
# и заметную долю пар с несеквенированным промежутком между R1 и R2 (mate-linked).
SIZE_SELECTION_TARGET = 250

_TRUTH_TAG = {"amplicon_with_primers": "amp", "vj_only": "vj"}[TRUTH_TEMPLATE_MODE]
assert UMI_COLLAPSE_MODE == "umi_consensus"
_TRUTH_TAG += "_umicons"
BRANCH_NAME = f"{BRANCH_NAME}_{_TRUTH_TAG}_in{LIBRARY_INPUT_NG:g}ng_rb{READ_BUDGET_MULTIPLIER:g}x_ss{SIZE_SELECTION_TARGET}"
SOURCE_STAGE = "post_annotation_filtered"

VOLUME = Path(os.environ.get("BCR_VOLUME", "/data/user/epishkin"))
if not (VOLUME / "results/PRJEB30386").exists():
    found = None
    for start in (Path.cwd().resolve(), Path("/Users/epishkin/workspace/bcr-assembler")):
        for candidate in (start, *start.parents):
            if (candidate / "results/PRJEB30386").exists() and (candidate / "scripts").is_dir():
                found = candidate
                break
        if found:
            break
    if not found:
        raise FileNotFoundError("Cannot locate bcr-assembler root; set BCR_VOLUME")
    VOLUME = found

DATASET = "PRJEB30386"
SOURCE_RUNS = ["ERR3004229", "ERR3004230", "ERR3004231", "ERR3004232"]
RUN_LOCUS = {
    "ERR3004229": "IGH",
    "ERR3004230": "IGH",
    "ERR3004231": "IGK",
    "ERR3004232": "IGL",
}
RUN_LIBRARY = {
    "ERR3004229": "IgM",
    "ERR3004230": "IgG",
    "ERR3004231": "IgK",
    "ERR3004232": "IgL",
}
MIXED_SAMPLE = "PRJEB30386_all_chains"
SAMPLES = [MIXED_SAMPLE]

DATASET_DIR = VOLUME / "results" / DATASET
POST_FILTER_DIR = DATASET_DIR / SOURCE_STAGE
TRUTH_AIRR_DIR = POST_FILTER_DIR / "airr_pass"
FILTERED_FASTQ_DIR = POST_FILTER_DIR / "fastq"
FILTER_SUMMARY_PATH = POST_FILTER_DIR / "filter_summary.json"

RAW_FASTQ_DIR = VOLUME / "raw" / DATASET
PR_FASTQ_DIR = DATASET_DIR / "pr_trimmed" / "fastq"
ORIGINAL_AIRR_DIR = DATASET_DIR / "annotation" / "igblast"

if not FILTER_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"Post-annotation filter summary not found: {FILTER_SUMMARY_PATH}. "
        "Run filter_human_post_annotation.ipynb first."
    )
FILTER_SUMMARY = json.loads(FILTER_SUMMARY_PATH.read_text())
FILTER_RULES = FILTER_SUMMARY["filters"]

assert FILTER_SUMMARY.get("dataset") == DATASET
assert FILTER_RULES["require_expected_locus"] == RUN_LOCUS
MIN_TEMPLATE_LENGTH = int(FILTER_RULES["min_vj_span_nt"])
MIN_V_IDENTITY = float(FILTER_RULES["min_v_identity_percent"])
MIN_J_IDENTITY = float(FILTER_RULES["min_j_identity_percent"])
MAX_V_SUPPORT = float(FILTER_RULES["max_v_support_evalue"])
MAX_J_SUPPORT = float(FILTER_RULES["max_j_support_evalue"])

TARGET_READ_LENGTH = 150

OUT_BASE = DATASET_DIR / "simulated" / BRANCH_NAME
TRUTH_DIR = OUT_BASE / "00_primary_truth"
PCR1_DIR = OUT_BASE / "01_pcr1"
FRAGMENTATION_DIR = OUT_BASE / "02_fragmentation"
PCR2_DIR = OUT_BASE / "03_pcr2"
ALLOCATION_DIR = OUT_BASE / "04_read_allocation"
FASTQ_NATIVE_DIR = OUT_BASE / "05_fastq_native"
FASTQ_DIR = OUT_BASE / "06_fastq_pe150"
MODEL_DIR = OUT_BASE / "model"
LOGS_DIR = OUT_BASE / "logs"
QC_DIR = OUT_BASE / "qc"
TEMPLATES_DIR = TRUTH_DIR
COUNTS_DIR = ALLOCATION_DIR

for d in (
    TRUTH_DIR, PCR1_DIR, FRAGMENTATION_DIR, PCR2_DIR, ALLOCATION_DIR,
    FASTQ_NATIVE_DIR, FASTQ_DIR, MODEL_DIR, LOGS_DIR, QC_DIR,
):
    d.mkdir(parents=True, exist_ok=True)

NPROC = 8
SEED = 42
FORCE = False
CLEAN_OLD_OUTPUTS = False
COMPRESS = True

# Дополнительные требования симуляции к эталонным данным после
# основной фильтрации аннотаций.
DROP_SEQUENCES_WITH_N = True
REQUIRE_VALID_20NT_UMI = True
STARTING_COPIES_MODE = "unique_umis"

# PCR1: стохастическое ветвление на каждом цикле.
PCR_CYCLES = 25
PCR_EFFICIENCY_MEAN = 0.85
PCR_EFFICIENCY_CONCENTRATION = 80.0
PCR_MAX_COPIES = 10**12

# Фрагментация и отбор библиотеки.
#
# Из PCR1-пула берётся конечная порция молекул, каждая молекула физически
# разрезается; до size-selection проверяется сохранение нуклеотидной массы.
SEQUENCE_TYPE = "amplicon"
FRAGMENTATION_MODEL = "ultrasonic_like_recursive_shearing"

# SIZE_SELECTION_TARGET задан в начале ячейки и входит в имя ветки.
SIZE_SELECTION_SD = 40
# ISS 2.0.1 simulate_read: assert read_length < len(template); NovaSeq profile read_length = 151 nt.
MIN_SEQUENCABLE_FRAGMENT_LENGTH = 152
# Перевод массы порции в молекулы: dsDNA ~650 г/моль на пару оснований.
DSDNA_G_PER_MOL_PER_BP = 650.0
AVOGADRO = 6.02214076e23
# Куски (start, end) с ожидаемым числом < порога на молекулу не учитываются;
# их суммарная доля пишется в fragment_length_model.tsv.
FRAGMENT_MODEL_MIN_EXPECTED = 1e-12

# Flux-Simulator-inspired mechanical/nebulisation abstraction.
# These are tunable simulation parameters, not physical constants.
ULTRASONIC_LIMITING_LENGTH = 200.0
ULTRASONIC_BREAKPOINT_SD_FRACTION = 0.20
ULTRASONIC_MAX_RECURSION_DEPTH = 6

FRAGMENTATION_MODEL_PROVENANCE = {
    "model": FRAGMENTATION_MODEL,
    "library_input": "LIBRARY_INPUT_NG of PCR1 product: molecules = ng / (PCR1-weighted mean length * 650 g/mol) * N_A, multinomial by PCR1 abundance",
    "library_input_ng": LIBRARY_INPUT_NG,
    "fragment_pool": "per template with reads: Poisson(aliquot copies * expected size-selected (start, end) fragments per molecule of its length), PCR2 per fragment species, reads multinomial by PCR2 copies; reads are split across templates by expected size-selected fragments",
    "break_probability": "exp(-limiting_length/current_fragment_length)",
    "limiting_length_nt": ULTRASONIC_LIMITING_LENGTH,
    "breakpoint_distribution": "truncated Gaussian around fragment midpoint",
    "breakpoint_sd_fraction": ULTRASONIC_BREAKPOINT_SD_FRACTION,
    "max_recursion_depth": ULTRASONIC_MAX_RECURSION_DEPTH,
    "size_selection": {
        "type": "Gaussian acceptance kernel",
        "target_nt": SIZE_SELECTION_TARGET,
        "sd_nt": SIZE_SELECTION_SD,
        "min_sequencable_nt": MIN_SEQUENCABLE_FRAGMENT_LENGTH,
    },
    "scientific_analog": "Flux Simulator nebulisation model; DOI 10.1093/nar/gks666",
    "scope_note": "approximate mechanical/ultrasonic-like shearing, not device-specific physics",
}

# PCR2: высокий предел предотвращает искусственное выравнивание численности.
LIBRARY_PCR_CYCLES = 10
LIBRARY_PCR_EFFICIENCY_MEAN = 0.85
LIBRARY_PCR_EFFICIENCY_CONCENTRATION = 80.0
LIBRARY_PCR_MAX_COPIES = 10**15

READ_BUDGET_MODE = "match_all_raw_pairs"
FIXED_READ_PAIRS = 500_000
MIXTURE_MODE = "observed_locus_depth"
RAW_PAIR_COUNTS = {
    "ERR3004229": 1_355_378,
    "ERR3004230": 1_153_931,
    "ERR3004231": 958_261,
    "ERR3004232": 1_085_144,
}

SEQUENCING_ERROR_MODEL_PROVENANCE = {
    "type": "InSilicoSeq bundled NovaSeq profile",
    "conditioned_on_post_annotation_filter": False,
}

print("SOURCE STAGE:", SOURCE_STAGE)
print("TRUTH AIRR:", TRUTH_AIRR_DIR)
print("BRANCH:", BRANCH_NAME)
print("OUT_BASE:", OUT_BASE)
print("filter rules:", FILTER_RULES)


In [ ]:
# ---------------------------------------------------------------------
# Очистка сгенерированных артефактов в целевой ветви
# ---------------------------------------------------------------------
# Каталог выходных данных:
#   results/PRJEB30386/simulated/insilicoseq/
#
# При FORCE=True и CLEAN_OLD_OUTPUTS=True содержимое целевых
# выходных каталогов удаляется и создаётся заново. Исходные данные
# за пределами OUT_BASE не изменяются.

def clean_previous_generated_outputs():
    if not (FORCE and CLEAN_OLD_OUTPUTS):
        print("Cleanup disabled.")
        return

    generated_dirs = [TRUTH_DIR,PCR1_DIR,FRAGMENTATION_DIR,PCR2_DIR,ALLOCATION_DIR,FASTQ_NATIVE_DIR,FASTQ_DIR,MODEL_DIR,QC_DIR,LOGS_DIR]

    for directory in generated_dirs:
        directory = Path(directory)
        if not directory.exists():
            continue
        for path in directory.iterdir():
            if path.is_file() or path.is_symlink():
                path.unlink()
            elif path.is_dir():
                shutil.rmtree(path)

    for directory in generated_dirs:
        Path(directory).mkdir(parents=True, exist_ok=True)

    print(f"Cleaned previous generated outputs under: {OUT_BASE}")

clean_previous_generated_outputs()


## 3. Вспомогательные функции

In [ ]:
def open_text(path, mode="rt"):
    return gzip.open(path, mode) if str(path).endswith(".gz") else open(path, mode)

def iter_fastq(path):
    with open_text(path, "rt") as h:
        while True:
            head = h.readline()
            if not head:
                return
            seq = h.readline().rstrip("\n\r")
            plus = h.readline()
            qual = h.readline()
            if not plus or not qual:
                raise ValueError(f"Truncated FASTQ: {path}")
            yield seq

def count_fastq(path):
    return sum(1 for _ in iter_fastq(path))

def iter_fasta(path):
    with open(path) as h:
        name, chunks = None, []
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = line[1:], []
            else:
                chunks.append(line)
        if name is not None:
            yield name, "".join(chunks)

def run_with_heartbeat(cmd, log_path, heartbeat=30, shell=False):
    t0 = time.time()
    with open(log_path, "w") as log_h:
        proc = subprocess.Popen(cmd, stdout=log_h, stderr=subprocess.STDOUT, text=True, shell=shell)
        label = cmd if shell else " ".join(map(str, cmd))
        print(f"[run] {label}\n  pid={proc.pid} log={log_path}")
        while proc.poll() is None:
            print(f"  running: elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(heartbeat)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}); see {log_path}")
    print(f"done in {(time.time()-t0)/60:.1f} min")

def sample_seed(sample, extra=0):
    return int(SEED + extra + sum((i + 1) * ord(ch) for i, ch in enumerate(sample)))

## 4. Построение эталона: ампликон PCR1 с учётом UMI

`post_annotation_filtered/airr_pass` служит эталонным входом. Условия фильтра повторно проверяются утверждениями. Дополнительно исключаются непригодные UMI и записи с N.

- **Ключ схлопывания** — (локус, наблюдаемая последовательность V…J); реальные SHM сохраняются.
- **`UMI_COLLAPSE_MODE = "umi_consensus"`**: каждый UMI — одна исходная молекула. Её V…J — поколоночный консенсус ридов UMI (модальный локус и длина, ≥ `MIN_UMI_CONSENSUS_READS` ридов, без ничьих); такая молекула **проверена** (`verified`). UMI без консенсуса (один рид, ничья) симулируются самым частым ридом; если он отличается одной заменой от проверенного консенсуса того же локуса, это тот же клон с ошибкой секвенирования и молекула становится копией консенсуса. Шаблон, у которого нет ни одной проверенной молекулы, помечается `verified=False`: он есть в библиотеке, но не входит в оценку сборщиков (его последовательность может нести ошибку секвенирования исходных данных). CDR3 (`junction`) записывается в `template_qc.tsv`.
- **`TRUTH_TEMPLATE_MODE = "amplicon_with_primers"`**: шаблон = V-leader праймер + склеенный рид (остаток leader, V…J, C-фрагмент) + revcomp(C-праймер). Праймеры были вырезаны `MaskPrimers --mode cut`; имена праймеров берутся из заголовков `pr_trimmed` (AssemblePairs их не перенёс), последовательности — из `technical_sequences.json`. UMI и Illumina-хвосты не включаются. Координаты V…J записаны в `vj_start`/`vj_end` (0-based, half-open) — именно этот интервал является эталоном для оценки сборщиков.


In [ ]:
_IUPAC = {"A": "A", "C": "C", "G": "G", "T": "T", "R": "AG", "Y": "CT", "S": "CG", "W": "AT",
          "K": "GT", "M": "AC", "B": "CGT", "D": "AGT", "H": "ACT", "V": "ACG", "N": "ACGT"}
_RC = str.maketrans("ACGT", "TGCA")

def _revcomp(s):
    return s.translate(_RC)[::-1]

def _truthy(x):
    return str(x).strip().lower() in {"t", "true", "1", "yes"}

def _falsey(x):
    return str(x).strip().lower() in {"f", "false", "0", "no"}

def _as_int(x):
    try:
        return int(x)
    except (TypeError, ValueError):
        return None

def _as_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

def _barcode(sequence_id):
    m = re.search(r"(?:^|\|)BARCODE=([^|\s]+)", sequence_id or "")
    return m.group(1) if m else None

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as src:
        for chunk in iter(lambda: src.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _find_technical_sequences():
    rel = Path("docs/datasets/PRJEB30386/technical_sequences.json")
    cwd = Path.cwd().resolve()
    for root in (VOLUME, cwd, *cwd.parents, Path("/Users/epishkin/workspace/bcr-assembler")):
        if (root / rel).exists():
            return root / rel
    raise FileNotFoundError(f"{rel} not found; needed for primer sequences")

def load_gene_specific_primers():
    """Gene-specific части праймеров библиотеки (без Illumina-хвоста и C-side UMI).

    Совпадают со словарями V_LEADER_PRIMERS / CONSTANT_PRIMERS в primer_trim_human.ipynb.
    """
    path = _find_technical_sequences()
    out = {}
    for p in json.loads(path.read_text())["supplemental_table_2_library_primers"]:
        tail = p["sequence"].split("TCTTCCGATCT", 1)
        if len(tail) != 2 or not tail[1]:
            continue  # Read2U и прочие чисто технические олиго
        out[p["name"]] = tail[1][20:] if "_Ig" in p["name"] else tail[1]
    return out, path

def _pr_trimmed_primer_names(run):
    """read id -> [V primer, C primer] из заголовков pr_trimmed (AssemblePairs их не перенёс)."""
    names = {}
    for mate, slot in ((1, 0), (2, 1)):
        with gzip.open(PR_FASTQ_DIR / f"{run}_{mate}.pr.fastq.gz", "rt") as h:
            for i, line in enumerate(h):
                if i % 4:
                    continue
                rid = line[1:].split(None, 1)[0].split("|", 1)[0]
                m = re.search(r"\|PRIMER=([^|\s]+)", line)
                names.setdefault(rid, [None, None])[slot] = m.group(1) if m else None
    return names

if TRUTH_TEMPLATE_MODE == "amplicon_with_primers":
    PRIMERS, TECHNICAL_SEQUENCES_PATH = load_gene_specific_primers()
    print("gene-specific primers:", len(PRIMERS), "from", TECHNICAL_SEQUENCES_PATH)
else:
    PRIMERS, TECHNICAL_SEQUENCES_PATH = {}, None

FILTER_SUMMARY_SHA256 = _sha256(FILTER_SUMMARY_PATH)
PROVENANCE_PATH = TRUTH_DIR / "source_provenance.json"

def _validate_post_filter_row(r, run):
    """Fail loudly if airr_pass no longer matches the recorded filter contract."""
    problems = []
    expected_locus = RUN_LOCUS[run]

    v_start = _as_int(r.get("v_sequence_start"))
    j_end = _as_int(r.get("j_sequence_end"))
    vgs = _as_int(r.get("v_germline_start"))
    vi = _as_float(r.get("v_identity"))
    ji = _as_float(r.get("j_identity"))
    vs = _as_float(r.get("v_support"))
    js = _as_float(r.get("j_support"))

    if v_start is None or j_end is None or j_end - v_start + 1 < MIN_TEMPLATE_LENGTH:
        problems.append("V-J span")
    if vgs != 1:
        problems.append("V 5-prime completeness")
    if not _truthy(r.get("complete_vdj")):
        problems.append("complete_vdj")
    if str(r.get("locus", "")).strip() != expected_locus:
        problems.append("locus")
    if not _truthy(r.get("productive")):
        problems.append("productive")
    if not _falsey(r.get("stop_codon")):
        problems.append("stop_codon")
    if vi is None or vi < MIN_V_IDENTITY:
        problems.append("v_identity")
    if ji is None or ji < MIN_J_IDENTITY:
        problems.append("j_identity")
    if vs is None or vs > MAX_V_SUPPORT:
        problems.append("v_support")
    if js is None or js > MAX_J_SUPPORT:
        problems.append("j_support")

    if problems:
        raise RuntimeError(
            f"{run} airr_pass violates filter_summary contract for "
            f"{r.get('sequence_id')}: {', '.join(problems)}"
        )

def _current_provenance():
    data = {
        "dataset": DATASET,
        "source_stage": SOURCE_STAGE,
        "truth_airr_dir": str(TRUTH_AIRR_DIR),
        "filtered_fastq_dir": str(FILTERED_FASTQ_DIR),
        "filter_summary": str(FILTER_SUMMARY_PATH),
        "filter_summary_sha256": FILTER_SUMMARY_SHA256,
        "filter_rules": FILTER_RULES,
        "simulation_specific_truth_rules": {
            "require_valid_20nt_umi": REQUIRE_VALID_20NT_UMI,
            "drop_sequences_with_N": DROP_SEQUENCES_WITH_N,
            "truth_template_mode": TRUTH_TEMPLATE_MODE,
            "primer_sequences": str(TECHNICAL_SEQUENCES_PATH) if TECHNICAL_SEQUENCES_PATH else None,
            "primer_names_source": str(PR_FASTQ_DIR) if PRIMERS else None,
            "flanks": "most frequent (V primer, leader rest, C stub, C primer) among member reads; IUPAC resolved by seeded RNG; UMI not included",
            "vj_coordinates": "template_qc.tsv vj_start/vj_end, 0-based half-open",
            "collapse_key": ["locus", "VJ_sequence"],
            "umi_collapse_mode": UMI_COLLAPSE_MODE,
            "min_umi_consensus_reads": MIN_UMI_CONSENSUS_READS,
            "error_correction": "unverified UMI molecule 1 substitution from a verified consensus (same locus and length) -> copy of that consensus",
            "verified": "template has >= 1 UMI whose consensus equals its V-J",
            "junction": "most frequent AIRR junction span among member reads, cut from template V-J",
            "starting_copies_mode": STARTING_COPIES_MODE,
        },
        "branch": BRANCH_NAME,
        "library_input_ng": LIBRARY_INPUT_NG,
        "read_budget_multiplier": READ_BUDGET_MULTIPLIER,
        "seed": SEED,
        "fragmentation_model": FRAGMENTATION_MODEL_PROVENANCE,
        "sequencing_error_model": SEQUENCING_ERROR_MODEL_PROVENANCE,
    }
    return data

def _check_cached_truth_provenance():
    if not PROVENANCE_PATH.exists():
        return False
    old = json.loads(PROVENANCE_PATH.read_text())
    if old.get("filter_summary_sha256") != FILTER_SUMMARY_SHA256:
        raise RuntimeError(
            "post_annotation_filtered/filter_summary.json changed since this "
            "simulation truth was built. Use FORCE=True and CLEAN_OLD_OUTPUTS=True "
            "or choose a new branch."
        )
    if old.get("simulation_specific_truth_rules") != _current_provenance()["simulation_specific_truth_rules"]:
        raise RuntimeError("Truth construction rules changed; use FORCE=True or a new branch.")
    return True

def _consensus(seqs):
    """Поколоночный консенсус ридов одной длины; None при ничьей в любой позиции."""
    if len(set(seqs)) == 1:
        return seqs[0]
    out = []
    for col in zip(*seqs):
        top = Counter(col).most_common(2)
        if len(top) > 1 and top[0][1] == top[1][1]:
            return None
        out.append(top[0][0])
    return "".join(out)

assert _consensus(["ACGT", "ACGT"]) == "ACGT"
assert _consensus(["ACGT", "ACGA", "ACGT"]) == "ACGT"
assert _consensus(["ACGT", "ACGA"]) is None


def _umi_molecule(reads):
    """reads одного UMI: [(locus, VJ, flank, junction_span, v_call, j_call)].

    Возвращает (locus, VJ молекулы, её риды, verified). Консенсус — по ридам модального
    локуса и модальной длины V…J; если их < MIN_UMI_CONSENSUS_READS или есть ничья,
    молекула = самый частый рид UMI (verified=False).
    """
    locus = Counter(r[0] for r in reads).most_common(1)[0][0]
    same = [r for r in reads if r[0] == locus]
    length = Counter(len(r[1]) for r in same).most_common(1)[0][0]
    group = [r for r in same if len(r[1]) == length]
    if len(group) >= MIN_UMI_CONSENSUS_READS:
        cons = _consensus([r[1] for r in group])
        if cons is not None:
            return locus, cons, group, True
    seq = max(Counter(r[1] for r in same).items(), key=lambda kv: (kv[1], kv[0]))[0]
    return locus, seq, [r for r in same if r[1] == seq], False

def _one_mismatch(a, b):
    d = 0
    for x, y in zip(a, b):
        if x != y:
            d += 1
            if d > 1:
                return False
    return True

_r = lambda s: ("IGH", s, None, None, "v", "j")
assert _umi_molecule([_r("ACGT"), _r("ACGA"), _r("ACGT")])[1:4:2] == ("ACGT", True)
assert _umi_molecule([_r("ACGT")])[3] is False
assert _umi_molecule([_r("ACGT"), _r("ACGA")])[3] is False


def build_mixed_templates(force=FORCE):
    sample = MIXED_SAMPLE
    out_fa = TEMPLATES_DIR / f"{sample}_templates.fasta"
    out_tsv = TEMPLATES_DIR / f"{sample}_template_qc.tsv"
    audit_path = QC_DIR / "source_truth_audit_by_run.tsv"
    umi_audit_path = QC_DIR / "umi_consensus_audit.tsv"

    if out_fa.exists() and out_tsv.exists() and audit_path.exists() and not force:
        _check_cached_truth_provenance()
        print("[skip] post-filter UMI-consensus templates exist")
        return

    assert REQUIRE_VALID_20NT_UMI, "umi_consensus требует UMI у каждой записи"
    umi_reads = defaultdict(list)  # run:UMI -> риды этой молекулы
    audit = []
    amplicon = TRUTH_TEMPLATE_MODE == "amplicon_with_primers"

    for run in SOURCE_RUNS:
        airr = TRUTH_AIRR_DIR / f"{run}.airr.tsv"
        filtered_fastq = FILTERED_FASTQ_DIR / f"{run}_filtered.fastq.gz"
        if not airr.exists():
            raise FileNotFoundError(f"Filtered AIRR missing: {airr}")
        if not filtered_fastq.exists():
            raise FileNotFoundError(f"Filtered FASTQ missing: {filtered_fastq}")
        primer_names = _pr_trimmed_primer_names(run) if amplicon else {}

        expected_passed = int(FILTER_SUMMARY["samples"][run]["passed"])
        a = {
            "source_run": run,
            "library": RUN_LIBRARY[run],
            "expected_locus": RUN_LOCUS[run],
            "expected_passed_from_filter_summary": expected_passed,
            "airr_pass_rows": 0,
            "simulation_eligible_rows": 0,
            "missing_barcode": 0,
            "invalid_barcode_length": 0,
            "non_acgt_barcode": 0,
            "missing_v_or_j_call": 0,
            "sequence_with_N": 0,
            "missing_primer_annotation": 0,
        }

        with open(airr, newline="") as h:
            for r in csv.DictReader(h, delimiter="\t"):
                a["airr_pass_rows"] += 1
                _validate_post_filter_row(r, run)

                full = (r.get("sequence") or "").upper()
                start = int(r["v_sequence_start"]) - 1
                end = int(r["j_sequence_end"])
                seq = full[start:end]

                if len(seq) < MIN_TEMPLATE_LENGTH:
                    raise RuntimeError(
                        f"{run}: cropped V-J sequence shorter than filter threshold "
                        f"for {r.get('sequence_id')}"
                    )

                if not r.get("v_call") or not r.get("j_call"):
                    a["missing_v_or_j_call"] += 1
                    continue

                if DROP_SEQUENCES_WITH_N and "N" in (full if amplicon else seq):
                    a["sequence_with_N"] += 1
                    continue

                bc = _barcode(r.get("sequence_id"))
                if not bc:
                    a["missing_barcode"] += 1
                    continue
                if len(bc) != 20:
                    a["invalid_barcode_length"] += 1
                    continue
                if set(bc.upper()) - set("ACGT"):
                    a["non_acgt_barcode"] += 1
                    continue

                if amplicon:
                    vp, cp = primer_names.get(r["sequence_id"].split("|", 1)[0], (None, None))
                    if vp not in PRIMERS or cp not in PRIMERS:
                        a["missing_primer_annotation"] += 1
                        continue
                    flank = (vp, full[:start], full[end:], cp)
                else:
                    flank = None

                # junction = sequence[cdr3_start-4 : cdr3_end+3] (AIRR, 1-based), в координатах V…J
                jspan = ((int(r["cdr3_start"]) - 4 - start, len(r["junction"]))
                         if r.get("junction") and r.get("cdr3_start") else None)
                a["simulation_eligible_rows"] += 1
                umi_reads[run + ":" + bc].append((r["locus"], seq, flank, jspan, r["v_call"], r["j_call"]))

        if a["airr_pass_rows"] != expected_passed:
            raise RuntimeError(
                f"{run}: airr_pass rows={a['airr_pass_rows']:,}, but "
                f"filter_summary passed={expected_passed:,}"
            )
        audit.append(a)
        del primer_names

    # Одна молекула на UMI: консенсус (verified) или самый частый рид (не проверен).
    molecules = []
    umi_stats = Counter()
    for umi, reads in umi_reads.items():
        locus, seq, members, verified = _umi_molecule(reads)
        umi_stats["consensus_verified" if verified else
                  ("single_read" if len(reads) == 1 else "no_consensus")] += 1
        molecules.append((umi.split(":", 1)[0], locus, seq, members, verified))
    del umi_reads

    # Непроверенная молекула, отличающаяся одной заменой от проверенного консенсуса того же
    # локуса, — тот же клон с ошибкой секвенирования (~15% ридов проверенных UMI несут замены):
    # она становится копией этого консенсуса (при нескольких — самого поддержанного).
    support = Counter((l, s) for _, l, s, _, v in molecules if v)
    index = defaultdict(list)
    for l, s in support:
        h = len(s) // 2
        index[(l, len(s), 0, s[:h])].append(s)
        index[(l, len(s), 1, s[h:])].append(s)

    def _verified_neighbour(locus, s):
        L, h, best = len(s), len(s) // 2, None
        for key, lo, hi in (((locus, L, 0, s[:h]), h, L), ((locus, L, 1, s[h:]), 0, h)):
            for p in index.get(key, ()):
                if p != s and _one_mismatch(s[lo:hi], p[lo:hi]) and (
                        best is None or (support[(locus, p)], p) > (support[(locus, best)], best)):
                    best = p
        return best

    groups = {}
    for run, locus, seq, members, verified in molecules:
        if not verified and (locus, seq) not in support:
            parent = _verified_neighbour(locus, seq)
            if parent:
                seq = parent
                umi_stats["unverified_corrected_to_consensus_1nt"] += 1
        g = groups.setdefault((locus, seq), {
            "observed_reads": 0, "umi_copies": 0, "verified_umis": 0,
            "runs": set(), "libraries": set(), "v_calls": set(), "j_calls": set(),
            "flanks": Counter(), "junctions": Counter(),
        })
        g["umi_copies"] += 1
        g["verified_umis"] += int(verified)
        g["observed_reads"] += len(members)
        g["runs"].add(run)
        g["libraries"].add(RUN_LIBRARY[run])
        for _, _, flank, jspan, v_call, j_call in members:
            g["v_calls"].add(v_call)
            g["j_calls"].add(j_call)
            if flank:
                g["flanks"][flank] += 1
            if jspan:
                g["junctions"][jspan] += 1
    del molecules
    print(f"UMI molecules: {dict(umi_stats)}; unique V-J={len(groups):,}; "
          f"verified V-J={sum(g['verified_umis'] > 0 for g in groups.values()):,}")

    ordered = sorted(
        groups.items(),
        key=lambda x: (
            x[0][0],
            -x[1]["umi_copies"],
            -x[1]["observed_reads"],
            x[0][1],
        ),
    )

    iupac_rng = np.random.default_rng(sample_seed(sample, 500))
    def _resolve(p):
        return "".join(c if c in "ACGT" else str(iupac_rng.choice(list(_IUPAC[c]))) for c in p)

    fields = [
        "template_id", "locus", "length", "vj_start", "vj_end", "v_primer", "c_primer",
        "observed_multiplicity", "unique_umis", "verified_umis", "verified", "junction",
        "source_runs", "libraries", "v_calls", "j_calls",
    ]
    with open(out_fa, "w") as fa, open(out_tsv, "w", newline="") as ts:
        w = csv.DictWriter(ts, fieldnames=fields, delimiter="\t")
        w.writeheader()
        for i, ((locus, seq), g) in enumerate(ordered, 1):
            tid = f"{sample}_{locus}_tpl_{i:07d}"
            if amplicon:
                vp, lead, cstub, cp = g["flanks"].most_common(1)[0][0]
                left = _resolve(PRIMERS[vp]) + lead
                template = left + seq + cstub + _revcomp(_resolve(PRIMERS[cp]))
            else:
                vp = cp = ""
                left, template = "", seq
            jo, jl = g["junctions"].most_common(1)[0][0] if g["junctions"] else (-1, 0)
            fa.write(f">{tid}\n{template}\n")
            w.writerow({
                "template_id": tid,
                "locus": locus,
                "length": len(template),
                "vj_start": len(left),
                "vj_end": len(left) + len(seq),
                "v_primer": vp,
                "c_primer": cp,
                "observed_multiplicity": g["observed_reads"],
                "unique_umis": g["umi_copies"],
                "verified_umis": g["verified_umis"],
                "verified": g["verified_umis"] > 0,
                "junction": seq[jo:jo + jl] if 0 <= jo and jo + jl <= len(seq) else "",
                "source_runs": ";".join(sorted(g["runs"])),
                "libraries": ";".join(sorted(g["libraries"])),
                "v_calls": ";".join(sorted(g["v_calls"])),
                "j_calls": ";".join(sorted(g["j_calls"])),
            })

    with open(audit_path, "w", newline="") as h:
        w = csv.DictWriter(h, fieldnames=list(audit[0]), delimiter="\t")
        w.writeheader()
        w.writerows(audit)
    with open(umi_audit_path, "w", newline="") as h:
        w = csv.DictWriter(h, fieldnames=["umi_class", "umis"], delimiter="\t")
        w.writeheader()
        w.writerows({"umi_class": k, "umis": v} for k, v in sorted(umi_stats.items()))

    PROVENANCE_PATH.write_text(json.dumps(_current_provenance(), indent=2) + "\n")
    print(
        f"post-filter mixed templates={len(ordered):,}; "
        f"loci={sorted({k[0] for k in groups})}"
    )

build_mixed_templates()


## 5. Сводка контроля качества дедупликации

In [ ]:
q=TEMPLATES_DIR/f"{MIXED_SAMPLE}_template_qc.tsv"; rows=list(csv.DictReader(open(q),delimiter="\t")); loci=sorted({r["locus"] for r in rows}); summary=[]
for locus in loci:
    x=[r for r in rows if r["locus"]==locus]; summary.append({"locus":locus,"unique_templates":len(x),"observed_reads":sum(int(r["observed_multiplicity"]) for r in x),"unique_umis":sum(int(r["unique_umis"]) for r in x),"verified_templates":sum(r["verified"]=="True" for r in x),"verified_umis":sum(int(r["verified_umis"]) for r in x)})
out=QC_DIR/"template_summary_by_locus.tsv"
with open(out,"w",newline="") as h:
    w=csv.DictWriter(h,fieldnames=list(summary[0]),delimiter="\t"); w.writeheader(); w.writerows(summary)
print(*summary,sep="\n"); print("wrote",out)


## 6. Готовая модель ISS NovaSeq

Встроенный KDE-профиль NovaSeq из ISS 2.0.1 создаёт риды длиной 151 nt. После стадий эталон → PCR1 → фрагментация → PCR2 генерируются PE151, затем последовательности и качества детерминированно обрезаются до PE150. Пользовательская модель не обучается.

In [ ]:
import iss
ISS_PROFILE_NAME="NovaSeq"; ISS_NATIVE_READ_LENGTH=151
profile_path=Path(iss.__file__).resolve().parent/"profiles"/ISS_PROFILE_NAME
with np.load(profile_path,allow_pickle=True) as z:
    assert int(z["read_length"])==ISS_NATIVE_READ_LENGTH
    print({"profile":ISS_PROFILE_NAME,"model":str(z["model"].item()),"native_read_length":int(z["read_length"]),"target":TARGET_READ_LENGTH})


## 7. Бюджет смешанного секвенирования

Один синтетический FASTQ содержит все локусы. Глубина по умолчанию равна сумме глубин четырёх исходных индексированных библиотек. Бюджеты локусов воспроизводят их вычислительное объединение; внутри локуса численность определяется UMI и моделью PCR. Нативное спаривание тяжёлой и лёгкой цепей не моделируется.


In [ ]:
def get_read_budget(sample):
    if READ_BUDGET_MODE=="fixed": return int(FIXED_READ_PAIRS)
    if READ_BUDGET_MODE=="match_all_raw_pairs":
        total=0
        for run in SOURCE_RUNS:
            n1=count_fastq(RAW_FASTQ_DIR/f"{run}_1.fastq.gz"); n2=count_fastq(RAW_FASTQ_DIR/f"{run}_2.fastq.gz")
            if n1!=n2: raise ValueError(f"{run}: mate count mismatch")
            total+=n1
        return int(round(total*READ_BUDGET_MULTIPLIER))
    raise ValueError(READ_BUDGET_MODE)
READ_BUDGETS={MIXED_SAMPLE:get_read_budget(MIXED_SAMPLE)}; print(READ_BUDGETS)


## 8. PCR1 для наблюдаемых шаблонов V…J


In [ ]:
def load_dedup_templates(sample):
    fa=TEMPLATES_DIR/f"{sample}_templates.fasta"; qc=TEMPLATES_DIR/f"{sample}_template_qc.tsv"; meta={r["template_id"]:r for r in csv.DictReader(open(qc),delimiter="\t")}
    return [{"template_id":tid,"sequence":seq,"length":len(seq),"vj_start":int(meta[tid].get("vj_start") or 0),"vj_end":int(meta[tid].get("vj_end") or len(seq)),"observed_multiplicity":int(meta[tid]["observed_multiplicity"]),"unique_umis":int(meta[tid]["unique_umis"]),"locus":meta[tid]["locus"],"verified":meta[tid]["verified"]=="True"} for tid,seq in iter_fasta(fa)]
def starting_copies(row):
    if STARTING_COPIES_MODE=="unique_umis": return max(1,row["unique_umis"])
    if STARTING_COPIES_MODE=="one_per_unique": return 1
    if STARTING_COPIES_MODE=="observed_multiplicity": return row["observed_multiplicity"]
    raise ValueError(STARTING_COPIES_MODE)
def branching_pcr(n0,efficiency,cycles,rng):
    n=int(n0)
    for _ in range(cycles):
        if n<=0:return 0
        n+=int(rng.binomial(n,efficiency))
        if n>=PCR_MAX_COPIES:return int(PCR_MAX_COPIES)
    return n
def simulate_pcr_pool(sample):
    rng=np.random.default_rng(sample_seed(sample,1000)); rows=load_dedup_templates(sample); mean=PCR_EFFICIENCY_MEAN; conc=PCR_EFFICIENCY_CONCENTRATION; alpha,beta=(mean*conc,(1-mean)*conc) if mean<1 else (None,None)
    for r in rows:
        p=1. if mean==1 else float(rng.beta(alpha,beta)); n0=starting_copies(r); r.update(starting_copies=n0,pcr_efficiency=p,pcr_copies=branching_pcr(n0,p,PCR_CYCLES,rng))
    return rows
_test_rng=np.random.default_rng(1); assert branching_pcr(1,1.,10,_test_rng)==2**10; assert branching_pcr(1,0.,10,_test_rng)==1; print("branching PCR smoke tests: OK")


## 9. Фрагментация: ultrasonic-like mechanical shearing

Это отдельная приближённая модель **ультразвуковой/механической фрагментации**, а не симуляция кавитации или конкретного прибора Covaris.

Каждая молекула порции из PCR1-пула проходит рекурсивный процесс:

- вероятность ещё одного разрыва зависит от текущей длины фрагмента: `p_break = exp(-lambda / L)`;
- потенциальная точка разрыва выбирается из дискретного Gaussian-распределения вокруг середины текущего фрагмента;
- дочерние фрагменты снова могут разрушаться, максимум до `ULTRASONIC_MAX_RECURSION_DEPTH`;
- после завершения shearing применяется тот же отдельный мягкий size-selection вокруг `SIZE_SELECTION_TARGET` nt (по умолчанию 250) (`SD=40`).

Такая абстракция основана на подходе Flux Simulator к mechanical/nebulisation fragmentation: size-dependent breaking probability + breakpoint distribution around molecular midpoint, а затем отдельный size selection (Griebel et al., 2012, DOI `10.1093/nar/gks666`). Параметры здесь масштабированы под короткие V–J ампликоны и должны рассматриваться как калибруемые, а не как физические константы ультразвука.

**Реализация (порция `LIBRARY_INPUT_NG`).** В фрагментацию идут 5 нг продукта PCR1 — тысячи копий каждой исходной молекулы. Перечислять миллиарды кусков не нужно: фрагментация не зависит от последовательности, поэтому для каждой длины шаблона один раз считается ожидаемое число кусков `(start, end)` на молекулу (с сохранением массы нуклеотидов). Затем только для шаблонов, получивших риды, пул кусков разыгрывается как Poisson(копии в порции × ожидаемое число), проходит PCR2 (эффективность своя у каждого вида куска), и риды шаблона распределяются по копиям после PCR2. Таблица `02_fragmentation/<sample>_fragment_length_model.tsv` — модель по длинам, `<sample>_library_summary.tsv` — сводка порции.


In [ ]:
def _size_selection_probability(length):
    """Probability that a physical fragment enters the PCR2 library (scalar or array)."""
    length = np.asarray(length, dtype=float)
    z = (length - SIZE_SELECTION_TARGET) / SIZE_SELECTION_SD
    return np.where(length < MIN_SEQUENCABLE_FRAGMENT_LENGTH, 0.0, np.exp(-0.5 * z * z))


def _ultrasonic_break_probability(length):
    """Flux-inspired size-dependent probability of another mechanical break."""
    length = int(length)
    if length <= 1:
        return 0.0
    return float(math.exp(-ULTRASONIC_LIMITING_LENGTH / length))


def _midpoint_break_weights(length):
    """Discrete Gaussian weights for inter-base breakpoints around midpoint."""
    positions = np.arange(1, int(length), dtype=float)
    if not len(positions):
        return positions.astype(int), np.asarray([], dtype=float)
    sigma = max(1.0, ULTRASONIC_BREAKPOINT_SD_FRACTION * float(length))
    z = (positions - float(length) / 2.0) / sigma
    weights = np.exp(-0.5 * z * z)
    weights /= weights.sum()
    return positions.astype(int), weights


FRAGMENT_ID_TAG = "us"


def expected_terminal_fragments(L):
    """Ожидаемое число концевых кусков (start, end) на одну молекулу длины L.

    Та же рекурсия, что для одной молекулы, но переносится ожидаемая масса: доля
    p_break куска длины l рвётся в точке с весом Гаусса вокруг середины, доля 1-p_break
    становится концевой; после ULTRASONIC_MAX_RECURSION_DEPTH раундов всё оставшееся
    концевое. Результат зависит только от L, поэтому считается один раз на длину.
    """
    n = L + 1
    active = np.zeros((n, n))
    active[0, L] = 1.0
    terminal = np.zeros((n, n))
    for _ in range(ULTRASONIC_MAX_RECURSION_DEPTH):
        idx, flow_all = [], []
        for l in range(1, L + 1):
            s = np.arange(0, L - l + 1)
            m = active[s, s + l]
            live = m > 0
            if not live.any():
                continue
            s, m = s[live], m[live]
            p = _ultrasonic_break_probability(l)
            terminal[s, s + l] += m * (1.0 - p)
            if p == 0:
                continue
            pos, w = _midpoint_break_weights(l)
            flow = np.outer(m * p, w).ravel()
            cut = (s[:, None] + pos[None, :]).ravel()
            left = np.repeat(s, len(pos))
            right = np.repeat(s + l, len(pos))
            idx += [left * n + cut, cut * n + right]
            flow_all += [flow, flow]
        if not idx:
            active = np.zeros((n, n))
            break
        active = np.bincount(np.concatenate(idx), np.concatenate(flow_all), n * n).reshape(n, n)
    terminal += active
    s, e = np.nonzero(terminal)
    return s, e, terminal[s, e]


# самопроверка: сохраняется масса нуклеотидов; молекула рвётся хотя бы на 1 кусок
_s, _e, _E = expected_terminal_fragments(450)
assert abs(float((_E * (_e - _s)).sum()) - 450) < 1e-6 and float(_E.sum()) >= 1
print("fragmentation model self-check: OK; expected pieces per 450-nt molecule:", round(float(_E.sum()), 3))


## 10. PCR2 для фрагментов

PCR2 выполняется внутри этапа библиотеки (раздел 11): стохастическое ветвление для каждого вида куска со своей эффективностью.


In [ ]:
def branching_pcr_vectorized(n0, efficiency, cycles, max_copies, rng):
    n = np.asarray(n0, dtype=np.int64).copy()
    eff = np.asarray(efficiency, dtype=np.float64)
    for _ in range(cycles):
        active = n > 0
        if not active.any():
            break
        new = np.zeros_like(n)
        new[active] = rng.binomial(n[active], eff[active])
        n = n + new
        np.minimum(n, max_copies, out=n)
    return n


_test_rng = np.random.default_rng(1)
_vec = branching_pcr_vectorized(
    np.array([1]), np.array([1.0]), 10, 10**15, _test_rng
)
assert int(_vec[0]) == 2**10
print("vectorized branching PCR smoke test: OK")


## 11. PCR1 → порция → фрагментация → PCR2 → распределение ридов


In [ ]:
def write_dict_rows(path, rows, fields):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", newline="") as h:
        w = csv.DictWriter(h, fieldnames=fields, delimiter="\t")
        w.writeheader()
        w.writerows({k: r[k] for k in fields} for r in rows)
    tmp.replace(path)

def read_dict_rows(path):
    with open(path, newline="") as h:
        return list(csv.DictReader(h, delimiter="\t"))

def run_pcr1_stage(sample, force=FORCE):
    out = PCR1_DIR / f"{sample}_pcr1_pool.tsv"
    if out.exists() and not force:
        print("[skip] PCR1", out)
        return
    rows = simulate_pcr_pool(sample)
    fields = [
        "template_id", "locus", "length", "observed_multiplicity",
        "unique_umis", "starting_copies", "pcr_efficiency", "pcr_copies",
    ]
    write_dict_rows(out, rows, fields)
    print("PCR1 rows:", len(rows), "->", out)


def locus_read_budgets(sample):
    """Пары ридов по локусам: доля локуса = доля его исходных библиотек в сырых парах."""
    assert MIXTURE_MODE == "observed_locus_depth"
    target = int(READ_BUDGETS[sample])
    depth = {l: sum(RAW_PAIR_COUNTS[r] for r in SOURCE_RUNS if RUN_LOCUS[r] == l) for l in ("IGH", "IGK", "IGL")}
    exact = {l: target * n / sum(depth.values()) for l, n in depth.items()}
    budget = {l: int(np.floor(v)) for l, v in exact.items()}
    for l in sorted(exact, key=lambda x: exact[x] - budget[x], reverse=True)[: target - sum(budget.values())]:
        budget[l] += 1
    return budget


_FRAGMENT_MODEL = {}

def retained_fragment_model(L):
    """(starts, ends, λ, кусков на молекулу, отброшенная доля λ, cumsum λ) для длины L;
    λ — ожидаемое число кусков (start, end) на молекулу, прошедших size-selection."""
    L = int(L)
    if L not in _FRAGMENT_MODEL:
        s, e, E = expected_terminal_fragments(L)
        if abs(float((E * (e - s)).sum()) - L) > 1e-6 * L:
            raise AssertionError(f"fragmentation nucleotide-mass conservation failed for L={L}")
        lam = E * _size_selection_probability(e - s)
        keep = lam >= FRAGMENT_MODEL_MIN_EXPECTED
        _FRAGMENT_MODEL[L] = (s[keep], e[keep], lam[keep], float(E.sum()), float(lam[~keep].sum()),
                              np.cumsum(lam[keep]))
    return _FRAGMENT_MODEL[L]


def run_library_stage(sample, force=FORCE):
    """PCR1-пул → порция LIBRARY_INPUT_NG → фрагментация + size-selection → PCR2 → риды.

    1. Порция: A молекул (масса → число молекул), мультиномиально по численности после PCR1.
    2. Риды делятся между шаблонами пропорционально ожидаемому числу отобранных кусков
       (копии в порции × λ его длины). Средний выход PCR2 от длины куска не зависит и
       сокращается; отклонение суммы по тысячам кусков шаблона от ожидания пренебрежимо.
    3. Только для шаблонов с ридами разыгрывается пул кусков: Poisson(копии × λ) по видам
       (start, end), PCR2 каждого вида со своей эффективностью, риды шаблона —
       мультиномиально по копиям после PCR2. Так тысячи копий на молекулу не требуют
       перечисления миллиардов кусков.
    """
    inp = PCR1_DIR / f"{sample}_pcr1_pool.tsv"
    out_counts = ALLOCATION_DIR / f"{sample}_read_counts.tsv"
    out_fa = ALLOCATION_DIR / f"{sample}_selected_fragments.fasta"
    out_qc = ALLOCATION_DIR / f"{sample}_allocation.tsv"
    summary_out = FRAGMENTATION_DIR / f"{sample}_library_summary.tsv"
    model_out = FRAGMENTATION_DIR / f"{sample}_fragment_length_model.tsv"
    if all(p.exists() for p in (out_counts, out_fa, out_qc, summary_out, model_out)) and not force:
        print("[skip] library", sample)
        return

    rows = read_dict_rows(inp)
    for r in rows:
        r["length"] = int(r["length"])
        r["starting_copies"] = int(r["starting_copies"])
        r["pcr_copies"] = int(r["pcr_copies"])
    seqs = dict(iter_fasta(TRUTH_DIR / f"{sample}_templates.fasta"))
    rng = np.random.default_rng(sample_seed(sample, 2000))

    lengths = np.array([r["length"] for r in rows])
    pcr1 = np.array([r["pcr_copies"] for r in rows], dtype=float)
    mean_len = float((pcr1 * lengths).sum() / pcr1.sum())
    n_input = int(LIBRARY_INPUT_NG * 1e-9 / (mean_len * DSDNA_G_PER_MOL_PER_BP) * AVOGADRO)
    aliquot = rng.multinomial(n_input, pcr1 / pcr1.sum())

    t0 = time.time()
    per_molecule = {int(L): float(retained_fragment_model(L)[2].sum()) for L in np.unique(lengths)}
    print(f"[{sample}] fragment model for {len(per_molecule)} lengths: {time.time() - t0:.0f} s", flush=True)
    weight = aliquot * np.array([per_molecule[int(L)] for L in lengths])

    loci = np.array([r["locus"] for r in rows])
    reads = np.zeros(len(rows), dtype=np.int64)
    budgets = locus_read_budgets(sample)
    for locus, n in budgets.items():
        if not n:
            continue
        idx = np.flatnonzero(loci == locus)
        if not len(idx) or weight[idx].sum() <= 0:
            raise RuntimeError(f"{sample}: no library molecules for locus {locus}")
        reads[idx] = rng.multinomial(n, weight[idx] / weight[idx].sum())

    mean, conc = LIBRARY_PCR_EFFICIENCY_MEAN, LIBRARY_PCR_EFFICIENCY_CONCENTRATION
    alpha, beta = mean * conc, (1 - mean) * conc
    fields = ["fragment_id", "template_id", "locus", "template_length", "fragment_start",
              "fragment_length", "fragment_input_copies", "library_pcr_efficiency",
              "library_pcr2_copies", "simulated_read_pairs"]
    tmp = {p: Path(str(p) + ".tmp") for p in (out_counts, out_fa, out_qc)}
    species_with_reads = fallback_templates = 0
    t0 = time.time()
    with open(tmp[out_counts], "w", newline="") as ch, open(tmp[out_fa], "w") as fh, \
         open(tmp[out_qc], "w", newline="") as qh:
        cw = csv.writer(ch, delimiter="\t")
        qw = csv.DictWriter(qh, fieldnames=fields, delimiter="\t")
        qw.writeheader()
        for done, i in enumerate(np.flatnonzero(reads), 1):
            r, m, n = rows[i], int(aliquot[i]), int(reads[i])
            s, e, lam, _, _, cdf = retained_fragment_model(r["length"])
            # Пул кусков шаблона: Poisson(m·λ) по видам (start, end). Если ожидаемых кусков
            # меньше, чем видов, то же распределение быстрее: K ~ Poisson(m·Σλ), виды — K выборок по λ.
            if m * cdf[-1] < len(lam):
                picks = np.searchsorted(cdf, rng.random(int(rng.poisson(m * cdf[-1]))) * cdf[-1], side="right")
                nz, k = np.unique(picks, return_counts=True)
            else:
                k = rng.poisson(m * lam)
                nz = np.flatnonzero(k)
                k = k[nz]
            if not len(k):
                # ponytail: у шаблона с 1–2 копиями в порции ни один кусок не прошёл отбор, а риды
                # по ожиданию есть — берём один кусок по λ; при реалистичной порции почти не бывает.
                nz = np.searchsorted(cdf, rng.random(1) * cdf[-1], side="right")
                k = np.ones(1, dtype=np.int64)
                fallback_templates += 1
            s, e = s[nz], e[nz]
            eff = np.ones(len(k)) if mean == 1 else rng.beta(alpha, beta, size=len(k))
            cp = branching_pcr_vectorized(k, eff, LIBRARY_PCR_CYCLES, LIBRARY_PCR_MAX_COPIES, rng)
            pairs = rng.multinomial(n, cp / cp.sum())
            seq = seqs[r["template_id"]]
            for j in np.flatnonzero(pairs):
                start, end = int(s[j]), int(e[j])
                fid = f"{r['template_id']}_{FRAGMENT_ID_TAG}_{start}_{end}"
                # InSilicoSeq readcount_file ожидает число ридов, поэтому пары умножаются на 2.
                cw.writerow([fid, int(pairs[j]) * 2])
                fh.write(f">{fid}\n{seq[start:end]}\n")
                qw.writerow({"fragment_id": fid, "template_id": r["template_id"], "locus": r["locus"],
                             "template_length": r["length"], "fragment_start": start,
                             "fragment_length": end - start, "fragment_input_copies": int(k[j]),
                             "library_pcr_efficiency": float(eff[j]), "library_pcr2_copies": int(cp[j]),
                             "simulated_read_pairs": int(pairs[j])})
                species_with_reads += 1
            if done % 200000 == 0:
                print(f"  [{sample}] templates with reads done: {done:,} ({(time.time() - t0) / 60:.1f} min)", flush=True)
    for p, t in tmp.items():
        t.replace(p)

    total_pairs = int(reads.sum())
    assert total_pairs == sum(budgets.values())
    starting_total = sum(r["starting_copies"] for r in rows)
    summary = {
        "sample": sample,
        "fragmentation_model": FRAGMENTATION_MODEL,
        "pcr1_templates": len(rows),
        "pcr1_molecules_total": int(pcr1.sum()),
        "starting_copies_total": starting_total,
        "library_input_ng": LIBRARY_INPUT_NG,
        "pcr1_weighted_mean_template_length": round(mean_len, 1),
        "library_input_molecules": n_input,
        "library_input_copies_per_starting_molecule": round(n_input / starting_total, 1),
        "library_input_templates": int((aliquot > 0).sum()),
        "expected_retained_fragments_per_input_molecule": round(float(weight.sum() / aliquot.sum()), 4),
        "templates_with_reads": int((reads > 0).sum()),
        "read_pairs": total_pairs,
        "fragment_species_with_reads": species_with_reads,
        "read_pairs_per_fragment_species": round(total_pairs / species_with_reads, 3),
        "templates_single_fragment_fallback": fallback_templates,
        "size_selection_target_nt": SIZE_SELECTION_TARGET,
        "size_selection_sd_nt": SIZE_SELECTION_SD,
        "min_sequencable_fragment_nt": MIN_SEQUENCABLE_FRAGMENT_LENGTH,
    }
    write_dict_rows(summary_out, [summary], list(summary))
    model_rows = []
    for L, cnt in zip(*np.unique(lengths, return_counts=True)):
        s, e, lam, pieces, pruned, _ = retained_fragment_model(L)
        model_rows.append({"template_length": int(L), "templates": int(cnt),
                           "expected_pieces_per_molecule": round(pieces, 4),
                           "expected_retained_per_molecule": round(float(lam.sum()), 6),
                           "mean_retained_length": round(float((lam * (e - s)).sum() / lam.sum()), 1),
                           "retained_species": len(lam), "pruned_expected": pruned})
    write_dict_rows(model_out, model_rows, list(model_rows[0]))
    print(f"[{sample}] library: {summary}")

for sample in SAMPLES:
    run_pcr1_stage(sample)
    run_library_stage(sample)


## 12. Сводка библиотеки


In [ ]:
library_summary = [read_dict_rows(FRAGMENTATION_DIR / f"{sample}_library_summary.tsv")[0] for sample in SAMPLES]
out = QC_DIR / "library_summary.tsv"
write_dict_rows(out, library_summary, list(library_summary[0]))
print(*library_summary, sep="\n")


## 12a. Восстановимость эталонных шаблонов (reconstructability)

Для каждого эталонного шаблона по фрагментам, получившим риды, определяется, что сборщик **в принципе** может восстановить:

| класс | смысл |
|---|---|
| `no_reads` | шаблон не попал в FASTQ |
| `partial_read_coverage` | часть V…J не прочитана ни одним ридом |
| `full_coverage_unlinked` | все позиции прочитаны, но есть точка, которую не пересекает ни один фрагмент: куски физически не связаны, их нельзя соединить ни перекрытием, ни mate-парами |
| `full_coverage_mate_linked_only` | каждая точка пересечена фрагментом, но хотя бы одна — только несеквенированным промежутком между R1 и R2 (проверка mate-pair extension TRUST4) |
| `full_coverage_read_overlap` | каждая точка лежит внутри рида с `RECONSTRUCT_MIN_FLANK_NT` с обеих сторон — сборка обычным перекрытием |

Эти классы — знаменатели для benchmark TRUST4: recall считается отдельно по каждому классу.


In [ ]:
# Класс восстановимости эталонного интервала V…J каждого шаблона по тому, что реально попало в FASTQ.
# ISS amplicon: R1 = [start, start+R), R2 = [end-R, end); вся вставка — физическая связь.
# Связь (bond) b между позициями b-1 и b:
#   read_overlap — b лежит внутри одного рида с >= K нт с каждой стороны
#                  (собирается обычным перекрытием);
#   mate_only    — b внутри фрагмента, но в несеквенированном промежутке между
#                  R1 и R2 (соединить может только mate-pair extension TRUST4);
#   unlinked     — ни один фрагмент не пересекает b: куски шаблона физически
#                  не связаны, соединить их нельзя без внешней информации.
RECONSTRUCT_MIN_FLANK_NT = 31  # ponytail: грубый аналог k-mer/min-overlap сборщика, калибруемый

def _bond_mask(L, intervals):
    d = np.zeros(L + 2, dtype=np.int32)
    for a, b in intervals:
        a, b = max(a, 1), min(b, L - 1)
        if a <= b:
            d[a] += 1
            d[b + 1] -= 1
    return np.cumsum(d)[: L + 1] > 0

def classify_template(L, frags, R=TARGET_READ_LENGTH, K=RECONSTRUCT_MIN_FLANK_NT, region=None):
    """frags: [(start, end)] фрагментов с ридами в координатах шаблона.

    region=(vj_start, vj_end): оценивается только эталонный интервал V…J
    (праймеры и leader/C-фланги не требуют сборки). Возвращает (class, read_covered_fraction).
    """
    reads = [iv for s, e in frags for iv in ((s, min(s + R, e)), (max(e - R, s), e))]
    if region is not None:
        a0, b0 = region
        clip = lambda ivs: [(max(a, a0) - a0, min(b, b0) - a0) for a, b in ivs if min(b, b0) > max(a, a0)]
        L, frags, reads = b0 - a0, clip(frags), clip(reads)
    if not frags:
        return "no_reads", 0.0
    cov = np.zeros(L + 1, dtype=np.int32)
    for a, b in reads:
        cov[a] += 1
        cov[b] -= 1
    covered = float((np.cumsum(cov)[:L] > 0).mean())
    if covered < 1.0:
        return "partial_read_coverage", covered
    by_frag = _bond_mask(L, [(s + K, e - K) for s, e in frags])
    by_read = _bond_mask(L, [(a + K, b - K) for a, b in reads])
    bonds = slice(K, L - K + 1)  # связи ближе K к краям шаблона не требуют соединения
    if not by_frag[bonds].all():
        return "full_coverage_unlinked", covered
    if not by_read[bonds].all():
        return "full_coverage_mate_linked_only", covered
    return "full_coverage_read_overlap", covered

# self-check: 400 нт, PE150
assert classify_template(400, [])[0] == "no_reads"
assert classify_template(400, [(0, 200), (200, 400)])[0] == "full_coverage_unlinked"
assert classify_template(300, [(0, 200), (100, 300)])[0] == "full_coverage_read_overlap"
assert classify_template(400, [(0, 400)])[0] == "partial_read_coverage"          # середина между R1 и R2 не прочитана
assert classify_template(300, [(0, 300)])[0] == "full_coverage_mate_linked_only"  # R1 и R2 стыкуются без перекрытия
assert classify_template(420, [(0, 420), (100, 320)])[0] == "full_coverage_mate_linked_only"
# фланги вне V…J не учитываются; фрагмент только в праймере = нет ридов в V…J
assert classify_template(500, [(0, 300), (200, 500)], region=(50, 450))[0] == "full_coverage_mate_linked_only"
assert classify_template(500, [(0, 40)], region=(50, 450))[0] == "no_reads"

def write_reconstructability_qc(sample):
    frags = defaultdict(list)
    for r in read_dict_rows(ALLOCATION_DIR / f"{sample}_allocation.tsv"):
        if int(r["simulated_read_pairs"]) > 0:
            s = int(r["fragment_start"])
            frags[r["template_id"]].append((s, s + int(r["fragment_length"])))
    rows, summary = [], defaultdict(lambda: defaultdict(int))
    for t in load_dedup_templates(sample):
        cls, cov = classify_template(t["length"], frags.get(t["template_id"], []),
                                     region=(t["vj_start"], t["vj_end"]))
        rows.append({"template_id": t["template_id"], "locus": t["locus"], "length": t["length"],
                     "unique_umis": t["unique_umis"],
                     "fragments_with_reads": len(frags.get(t["template_id"], [])),
                     "read_covered_fraction": round(cov, 4), "reconstructability": cls, "verified": t["verified"]})
        if t["verified"]:  # знаменатели оценки сборщиков — только проверенные шаблоны
            summary[t["locus"]][cls] += 1
    write_dict_rows(QC_DIR / f"{sample}_template_reconstructability.tsv", rows, list(rows[0]))
    classes = ["no_reads", "partial_read_coverage", "full_coverage_unlinked",
               "full_coverage_mate_linked_only", "full_coverage_read_overlap"]
    out = []
    for locus in sorted(summary) + ["ALL"]:
        c = {k: (sum(summary[l][k] for l in summary) if locus == "ALL" else summary[locus][k]) for k in classes}
        n = sum(c.values())
        out.append({"sample": sample, "locus": locus, "templates": n, **c,
                    **{f"{k}_frac": round(c[k] / n, 4) for k in classes}})
    write_dict_rows(QC_DIR / f"{sample}_reconstructability_summary.tsv", out, list(out[0]))
    print(*out, sep="\n")

for sample in SAMPLES:
    write_reconstructability_qc(sample)


## 13. Генерация ридов точной длины PE150


In [ ]:
def crop_fastq(in_path,out_path,length):
    tmp=Path(str(out_path)+".tmp")
    with open_text(in_path,"rt") as src,gzip.open(tmp,"wt") as dst:
        n=0
        while True:
            h=src.readline()
            if not h: break
            s=src.readline().rstrip("\n\r"); p=src.readline(); q=src.readline().rstrip("\n\r")
            if len(s)<length or len(q)<length: raise ValueError(f"short read in {in_path}: {len(s)}")
            dst.write(h); dst.write(s[:length]+"\n"); dst.write(p); dst.write(q[:length]+"\n"); n+=1
    tmp.replace(out_path); return n

def run_iss_generate(sample,force=FORCE):
    fa=ALLOCATION_DIR/f"{sample}_selected_fragments.fasta"; counts=ALLOCATION_DIR/f"{sample}_read_counts.tsv"; native_prefix=FASTQ_NATIVE_DIR/sample; final_prefix=FASTQ_DIR/sample; ext=".fastq.gz"
    nr1=Path(str(native_prefix)+f"_R1{ext}"); nr2=Path(str(native_prefix)+f"_R2{ext}"); r1=Path(str(final_prefix)+f"_R1{ext}"); r2=Path(str(final_prefix)+f"_R2{ext}")
    if r1.exists() and r2.exists() and not force: print("[skip]",sample); return
    cmd=["iss","generate","--genomes",str(fa),"--readcount_file",str(counts),"--sequence_type",SEQUENCE_TYPE,"--model",ISS_PROFILE_NAME,"--cpus",str(NPROC),"--output",str(native_prefix),"--seed",str(sample_seed(sample,5000)),"--compress"]
    run_with_heartbeat(cmd,LOGS_DIR/f"{sample}_iss_novaseq.log")
    n1=crop_fastq(nr1,r1,TARGET_READ_LENGTH); n2=crop_fastq(nr2,r2,TARGET_READ_LENGTH); assert n1==n2==READ_BUDGETS[sample]
    print("NovaSeq native PE151 -> exact PE150:",n1,"pairs")
for sample in SAMPLES: run_iss_generate(sample)


## 14. Итоговая проверка


In [ ]:
final_rows=[]
for sample in SAMPLES:
    r1=FASTQ_DIR/f"{sample}_R1.fastq.gz"; r2=FASTQ_DIR/f"{sample}_R2.fastq.gz"; n1=count_fastq(r1); n2=count_fastq(r2)
    lengths1=sorted({len(s) for s in iter_fastq(r1)}); lengths2=sorted({len(s) for s in iter_fastq(r2)})
    final_rows.append({"branch":BRANCH_NAME,"sample":sample,"expected_pairs":READ_BUDGETS[sample],"R1_reads":n1,"R2_reads":n2,"R1_lengths":",".join(map(str,lengths1)),"R2_lengths":",".join(map(str,lengths2)),"valid":n1==n2==READ_BUDGETS[sample] and lengths1==lengths2==[TARGET_READ_LENGTH]})
out=QC_DIR/"final_qc.tsv"; write_dict_rows(out,final_rows,list(final_rows[0])); assert all(r["valid"] for r in final_rows); print(*final_rows,sep="\n")


## Интерпретация и научные ограничения

- Биологический эталон формируется из AIRR-записей после фильтрации аннотаций.
- Сохраняется наблюдаемая последовательность V…J; germline-последовательность её не заменяет, поэтому реальные SHM остаются в данных.
- Шаблон — физический ампликон PCR1 (V-leader праймер + склеенный рид + C-праймер); эталон для оценки сборки — интервал `vj_start…vj_end`. UMI и Illumina-хвосты в шаблон не включены; вырожденные позиции праймеров разрешены случайно, но воспроизводимо.
- Каждый UMI — одна молекула; её V…J — консенсус ридов UMI. Шаблоны без проверенного консенсуса (≈80% UMI прочитаны одним ридом) симулируются, но в оценку сборщиков не входят.
- Численность UMI задаёт исходные копии. Записи с отсутствующим или некорректным UMI и неоднозначными N исключаются и учитываются отдельно.
- PCR1 и PCR2 моделируются стохастическим ветвлением. Замены полимеразы, химеры, истощение реагентов и изменение эффективности по циклам явно не моделируются.
- `observed_locus_depth` сохраняет наблюдаемую долю IGH/IGK/IGL.
- Встроенная модель InSilicoSeq NovaSeq является общим профилем платформы: ISS создаёт PE151, после чего оба рида детерминированно обрезаются до PE150.
- Смешанный IGH+IGK+IGL набор является вычислительным тестом без нативного спаривания тяжёлой и лёгкой цепей.
- В фрагментацию идёт `LIBRARY_INPUT_NG` нг продукта PCR1 (тысячи копий на исходную молекулу); модель кусков по длине проверяется на сохранение массы нуклеотидов. Выход PCR2 шаблона берётся по ожиданию (сумма по тысячам кусков), разброс PCR2 между кусками моделируется.
- Фрагментация использует приближённую ultrasonic-like recursive mechanical-shearing модель; она не претендует на моделирование конкретного прибора.
